# UD05 · Notebook 8 — Sistema experto como controlador de un proceso

**Objetivo**: implementar un **controlador experto** que regule la temperatura de una sala,
definiendo las especificaciones de respuesta (CE d) y observando cómo influye en el comportamiento
del sistema (CE e).

Es **práctica**: se trabaja en clase y no se entrega ni puntúa. Es el último de la unidad y da por
supuestos los notebooks anteriores, en particular el control difuso del notebook 7.

## Antes de empezar

Python 3.10+ con `experta`. Recuerda el **parche de compatibilidad** (`collections.Mapping`
desapareció en Python 3.10):

```python
import collections, collections.abc
if not hasattr(collections, 'Mapping'):
    collections.Mapping = collections.abc.Mapping
    collections.Iterable = collections.abc.Iterable
    collections.MutableMapping = collections.abc.MutableMapping
```

## Fase 1 — La planta

Escribe el modelo de la sala: dada una temperatura y una potencia, devuelve la temperatura del
siguiente instante. Tiene inercia térmica y pierde calor hacia el exterior.

In [ ]:
def simular_planta(temp, potencia, dt=1.6, inercia=0.05, exterior=15.0):
    """Modelo simple de una sala con inercia termica."""
    # TU CÓDIGO
    ...

**✏️ Respuesta**: ¿qué representa cada parámetro? ¿Qué pasaría con `inercia = 0`?

*(escribe aquí)*

## Fase 2 — El controlador experto

Cuatro reglas sobre el **error** (consigna − temperatura): error grande → máxima potencia, error
medio → media, error pequeño → baja, error negativo → apagar.

### Dos trampas que te van a costar tiempo

**1. El condicional va envuelto en `P(...)`.** `experta` compara el valor de un campo de un `Fact`
por **igualdad**, no lo evalúa: si escribes `Fact(error=lambda e: e > 3)`, la regla busca un hecho
cuyo `error` sea literalmente esa función, y nunca lo encuentra. `P(...)` es el *constraint* de
predicado de `experta`: le dice al motor «aplica esta función al valor del campo y compara el
resultado con verdadero». Sin `P()`, el controlador se queda **congelado con potencia 0** para
siempre, y no parece un error evidente porque no lanza ninguna excepción.

**2. El motor necesita `reset()` en cada paso**, o los `DefFacts` no se recargan.

Con estos parámetros, la temperatura entra en la banda 20,5-21,5 ºC hacia el paso 5 y se estabiliza
en torno a **21,4 ºC**, dentro de la especificación.

In [ ]:
%pip install experta

# experta (2019) rompe en Python 3.10+ porque `collections.Mapping` se eliminó.
# Parche de tres líneas antes de importar (ya documentado en la UD00):
import collections, collections.abc
if not hasattr(collections, 'Mapping'):
    collections.Mapping = collections.abc.Mapping
    collections.Iterable = collections.abc.Iterable
    collections.MutableMapping = collections.abc.MutableMapping

from experta import *


class ControladorClima(KnowledgeEngine):
    def __init__(self, setpoint=21.0):
        super().__init__()
        self.setpoint = setpoint
        self.potencia = 0.0

    # TU CÓDIGO: DefFacts, la regla que lee el sensor y las cuatro reglas de potencia
    ...

    def paso(self, temp):
        self._temp = temp
        self.reset()
        self.run()
        return self.potencia


ctrl = ControladorClima(setpoint=21.0)
temp = 15.0
for i in range(200):
    potencia = ctrl.paso(temp)
    temp = simular_planta(temp, potencia)
    if i % 20 == 0:
        print(f"t={i:3d}: temp={temp:.2f}  potencia={potencia:.2f}")

## Fase 3 — Mide las especificaciones de respuesta

Añade el código que mida las tres cosas que definen la calidad de la respuesta.

In [ ]:
# TU CÓDIGO: mide error en regimen permanente, tiempo de asentamiento y sobreimpulso
# - error: temperatura final menos setpoint
# - asentamiento: en que paso entra y se queda en la banda 20,5-21,5 C
# - sobreimpulso: si supera 21,5 C en algun momento, y cuanto
...

**✏️ Respuesta**: rellena la tabla con **tus** números.

| Especificación | Valor |
|---|---|
| Error en régimen permanente | |
| Tiempo de asentamiento (pasos) | |
| Sobreimpulso máximo | |

## Fase 4 — Añade una perturbación

En el paso 100, baja la temperatura exterior a 5 ºC: alguien ha abierto una ventana.

In [ ]:
# TU CÓDIGO: repite la simulacion con la perturbacion en el paso 100
...

**✏️ Respuesta**: ¿mantiene la temperatura en la banda? ¿Cuánto tarda en recuperarse?

*(escribe aquí)*

## Fase 5 — Analiza la sensibilidad

Cambia el umbral de la regla de potencia media (de 3 a 1,5) y repite las medidas de la Fase 3.

In [ ]:
# TU CÓDIGO
...

**✏️ Respuesta**: la misma tabla, **antes y después** del cambio. ¿Qué le pasa al
sobreimpulso? ¿Y al tiempo de asentamiento?

| Especificación | Umbral 3 | Umbral 1,5 |
|---|---|---|
| Error en régimen permanente | | |
| Tiempo de asentamiento | | |
| Sobreimpulso | | |

## Fase 6 — Compara con un PID

**✏️ Respuesta**: describe cómo respondería un PID bien sintonizado frente a este controlador
experto, y **cuándo usarías cada uno**. No hace falta implementarlo.

*(escribe aquí)*

## Cierre

**✏️ Respuesta**: ¿qué relación tiene un sistema experto con los sistemas híbridos reglas/datos y
con la lógica difusa? ¿Cuándo usarías cada uno?

*(escribe aquí)*

## Qué tienes que tener al terminar

La simulación y las explicaciones de las seis fases, en este mismo notebook.

| Fase | Evidencia mínima |
|---|---|
| 1 | La planta simulada y qué representa cada parámetro |
| 2 | El controlador experto funcionando, con las reglas que se disparan en cada tramo |
| 3 | La **tabla de especificaciones**: error en régimen permanente, tiempo de asentamiento y sobreimpulso |
| 4 | La respuesta ante la perturbación, y si se mantiene en la banda |
| 5 | La misma tabla **antes y después** de cambiar el umbral |
| 6 | La comparación razonada con un PID: cuándo usarías cada uno |

Las soluciones no se publican: se corrigen y comentan en clase.